# MVP - Classificação de Faixa de Preço de Combustível\n
\n
## Contexto\n
Este notebook utiliza o dataset **Global Fuel Prices: 100 Years (1924-2024)** para resolver um problema de classificação supervisionada: prever `price_tier`.

## 1. Carga de dados por URL (requisito do MVP)\n
Substitua a URL abaixo pelo RAW do seu repositório GitHub.

In [ ]:
import pandas as pd\n
\n
DATA_URL = 'https://raw.githubusercontent.com/SEU_USUARIO/SEU_REPO/main/fuel_prices/all_countries_combined.csv'\n
df = pd.read_csv(DATA_URL)\n
df.head()

## 2. Preparação de dados (holdout + transformação + pipelines)

In [ ]:
from sklearn.model_selection import train_test_split\n
\n
features = [\n
    'year','region','subsidy_regime','is_oil_producer',\n
    'crude_oil_usd_per_barrel','tax_pct_of_pump_price','gasoline_real_2024usd'\n
]\n
target = 'price_tier'\n
\n
df = df.dropna(subset=features + [target]).copy()\n
X = df[features]\n
y = df[target]\n
\n
X_train, X_test, y_train, y_test = train_test_split(\n
    X, y, test_size=0.2, random_state=42, stratify=y\n
)\n
\n
X_train.shape, X_test.shape

## 3. Modelagem (KNN, Árvore, Naive Bayes e SVM) + otimização

In [ ]:
from sklearn.compose import ColumnTransformer\n
from sklearn.pipeline import Pipeline\n
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler\n
from sklearn.model_selection import GridSearchCV\n
from sklearn.metrics import accuracy_score\n
from sklearn.neighbors import KNeighborsClassifier\n
from sklearn.tree import DecisionTreeClassifier\n
from sklearn.naive_bayes import GaussianNB\n
from sklearn.svm import SVC\n
\n
num_features = ['year','is_oil_producer','crude_oil_usd_per_barrel','tax_pct_of_pump_price','gasoline_real_2024usd']\n
cat_features = ['region','subsidy_regime']\n
\n
base_pre = ColumnTransformer([\n
    ('num', Pipeline([('scaler', StandardScaler())]), num_features),\n
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),\n
])\n
\n
candidates = {\n
    'KNN': (KNeighborsClassifier(), {'clf__n_neighbors': [3,5,7,9]}),\n
    'DecisionTree': (DecisionTreeClassifier(random_state=42), {'clf__max_depth': [3,5,8,None], 'clf__min_samples_split': [2,5,10]}),\n
    'NaiveBayes': (GaussianNB(), {}),\n
    'SVM': (SVC(random_state=42), {\n
        'pre__num__scaler': [StandardScaler(), MinMaxScaler(), RobustScaler()],\n
        'clf__C': [0.5,1,5,10],\n
        'clf__kernel': ['linear','rbf']\n
    })\n
}\n
\n
results = []\n
best_model = None\n
best_acc = -1\n
\n
for name, (clf, grid) in candidates.items():\n
    pipe = Pipeline([('pre', base_pre), ('clf', clf)])\n
    gs = GridSearchCV(pipe, grid, cv=5, scoring='accuracy', n_jobs=-1)\n
    gs.fit(X_train, y_train)\n
    pred = gs.best_estimator_.predict(X_test)\n
    acc = accuracy_score(y_test, pred)\n
    results.append((name, acc, gs.best_params_))\n
    if acc > best_acc:\n
        best_acc = acc\n
        best_model = gs.best_estimator_\n
\n
results

## 4. Exportação do modelo vencedor

In [ ]:
import joblib\n
\n
joblib.dump(best_model, 'fuel_price_tier_model.pkl')\n
best_acc

## 5. Análise final\n
Fuel-Insight-MVP é uma aplicação full stack com foco comercial que prevê faixas de preço de combustíveis usando machine learning. A plataforma oferece autenticação (login/cadastro), navegação SPA, predição unitária e em lote, analytics por região/país, alertas de risco, dashboard executivo com KPIs e módulo de mercado comparativo com gráfico (Brent, WTI, USD/BRL e outros ativos).

Seu objetivo é apoiar decisões estratégicas de preço e risco no setor de combustíveis, transformando dados históricos em insights práticos para operação, planejamento e governança de modelos (MLOps), com registro, ativação e monitoramento de versões do modelo.
\n

